## Téléchargement des images et collecte des métadonnées

Ce notebook permet de collecter automatiquement des images libres de droit à l'aide d'une requête SPARQL sur Wikidata. Les images sont enregistrées localement dans un dossier dédié, et leurs métadonnées techniques (format, dimensions, orientation, date de prise de vue, modèle d'appareil photo) sont extraites et sauvegardées dans un fichier CSV.

Ce script répond à la première étape du projet : collecte de données.

Les principales opérations réalisées sont :
- Construction et exécution d'une requête SPARQL pour récupérer des images associées à des villes.
- Téléchargement des images (format JPG ou PNG) dans un dossier local.
- Extraction des métadonnées via la bibliothèque Pillow (EXIF).
- Enregistrement des informations dans un fichier CSV pour analyse ultérieure.

Les images utilisées proviennent de Wikidata, et sont donc sous licence libre.

In [ ]:
import os
import csv
import requests
from PIL import Image, ExifTags
from io import BytesIO
from tqdm import tqdm

# Configuration
SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
OUTPUT_CSV = "../metadata/images_metadata.csv"
IMAGE_DIR = "../images"
HEADERS = {"User-Agent": "MassiveDataProject/1.0 (cmvilleroy@gmail.com)"}

# Création des dossiers si nécessaires
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

def build_sparql_query(limit=100):
    return f"""
    SELECT DISTINCT ?ville ?villeLabel ?pays ?paysLabel ?image WHERE {{
      ?ville wdt:P31 wd:Q1549591;
             wdt:P17 ?pays;
             wdt:P18 ?image.
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "fr". }}
    }}
    LIMIT {limit}
    """

def run_sparql_query(query):
    response = requests.get(SPARQL_ENDPOINT, params={"query": query, "format": "json"}, headers=HEADERS)
    response.raise_for_status()
    return response.json()["results"]["bindings"]

def download_image(url, filename):
    try:
        response = requests.get(url, stream=True, timeout=10, headers=HEADERS)
        response.raise_for_status()
        image_path = os.path.join(IMAGE_DIR, filename)
        if not os.path.exists(image_path):
            with open(image_path, "wb") as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
        return image_path
    except Exception as e:
        print(f"Erreur lors du téléchargement : {url} | {e}")
        return None

def get_image_metadata(image_path):
    metadata = {
        "format": None,
        "width": None,
        "height": None,
        "orientation": None,
        "capture_date": None,
        "device": None
    }
    try:
        with Image.open(image_path) as img:
            metadata["format"] = img.format
            metadata["width"], metadata["height"] = img.size
            metadata["orientation"] = (
                "portrait" if img.height > img.width
                else "landscape" if img.width > img.height
                else "square"
            )
            exif_data = img._getexif()
            if exif_data:
                exif = {ExifTags.TAGS.get(k, k): v for k, v in exif_data.items()}
                metadata["capture_date"] = exif.get("DateTimeOriginal")
                make = exif.get("Make", "")
                model = exif.get("Model", "")
                metadata["device"] = f"{make} {model}".strip()
    except Exception as e:
        print(f"Erreur lecture métadonnées {image_path} : {e}")
    return metadata

def get_commons_metadata(file_url):
    filename = os.path.basename(file_url)
    api_url = "https://commons.wikimedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "titles": f"File:{filename}",
        "prop": "imageinfo",
        "iiprop": "timestamp|user|url|extmetadata"
    }
    try:
        response = requests.get(api_url, params=params, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
        page = next(iter(data["query"]["pages"].values()))
        info = page.get("imageinfo", [{}])[0]
        meta = info.get("extmetadata", {})
        return {
            "author": meta.get("Artist", {}).get("value", ""),
            "license": meta.get("LicenseShortName", {}).get("value", ""),
            "description": meta.get("ImageDescription", {}).get("value", "")
        }
    except Exception as e:
        print(f"Erreur Wikimedia metadata : {e}")
        return {
            "author": "",
            "license": "",
            "description": ""
        }

def process_images(limit=100):
    query = build_sparql_query(limit)
    results = run_sparql_query(query)

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = [
            "ville", "pays", "image_url", "image_filename",
            "format", "width", "height", "orientation",
            "capture_date", "device",
            "author", "license", "description"
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for entry in tqdm(results, desc="Téléchargement et extraction"):
            ville = entry["villeLabel"]["value"]
            pays = entry["paysLabel"]["value"]
            image_url = entry["image"]["value"]
            image_filename = os.path.basename(image_url).split("?")[0]

            local_path = download_image(image_url, image_filename)
            if local_path:
                exif_meta = get_image_metadata(local_path)
                commons_meta = get_commons_metadata(image_url)
                writer.writerow({
                    "ville": ville,
                    "pays": pays,
                    "image_url": image_url,
                    "image_filename": image_filename,
                    "format": exif_meta["format"],
                    "width": exif_meta["width"],
                    "height": exif_meta["height"],
                    "orientation": exif_meta["orientation"],
                    "capture_date": exif_meta["capture_date"],
                    "device": exif_meta["device"],
                    "author": commons_meta["author"],
                    "license": commons_meta["license"],
                    "description": commons_meta["description"]
                })

# Lancement
process_images(limit=100)

Téléchargement et extraction:  22%|██▏       | 221/1000 [03:10<11:15,  1.15it/s]

Erreur lors du téléchargement : http://commons.wikimedia.org/wiki/Special:FilePath/%D0%90%D1%80%D1%85%D0%B8%D1%82%D0%B5%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B9%20%D0%B0%D0%BD%D1%81%D0%B0%D0%BC%D0%B1%D0%BB%D1%8C%20%D0%A2%D1%80%D0%B8%D1%84%D0%BE%D0%BD%D0%BE%D0%B2%D0%B0%20%D0%BC%D0%BE%D0%BD%D0%B0%D1%81%D1%82%D1%8B%D1%80%D1%8F%2C%20%D0%B2%D0%B8%D0%B4%20%D1%81%20%D0%BE%D0%B7%D0%B5%D1%80%D0%B0%20%D0%BE%D1%81%D0%B5%D0%BD%D1%8C%D1%8E.jpg | [Errno 63] File name too long: '../images/%D0%90%D1%80%D1%85%D0%B8%D1%82%D0%B5%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B9%20%D0%B0%D0%BD%D1%81%D0%B0%D0%BC%D0%B1%D0%BB%D1%8C%20%D0%A2%D1%80%D0%B8%D1%84%D0%BE%D0%BD%D0%BE%D0%B2%D0%B0%20%D0%BC%D0%BE%D0%BD%D0%B0%D1%81%D1%82%D1%8B%D1%80%D1%8F%2C%20%D0%B2%D0%B8%D0%B4%20%D1%81%20%D0%BE%D0%B7%D0%B5%D1%80%D0%B0%20%D0%BE%D1%81%D0%B5%D0%BD%D1%8C%D1%8E.jpg'


Téléchargement et extraction:  29%|██▉       | 289/1000 [04:12<09:05,  1.30it/s]

Erreur lors du téléchargement : http://commons.wikimedia.org/wiki/Special:FilePath/%D0%9F%D1%80%D0%BE%D1%81%D0%BF%D0%B5%D0%BA%D1%82%20%D0%9B%D0%B5%D0%BD%D0%B8%D0%BD%D0%B0%20%D0%B2%D0%BE%D0%B7%D0%BB%D0%B5%20%D0%BE%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%BA%D0%B8%20%C2%AB%D0%9A%D0%B8%D0%BD%D0%BE%D1%82%D0%B5%D0%B0%D1%82%D1%80%20%D0%A6%D0%B5%D0%BD%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9%C2%BB.jpg | [Errno 63] File name too long: '../images/%D0%9F%D1%80%D0%BE%D1%81%D0%BF%D0%B5%D0%BA%D1%82%20%D0%9B%D0%B5%D0%BD%D0%B8%D0%BD%D0%B0%20%D0%B2%D0%BE%D0%B7%D0%BB%D0%B5%20%D0%BE%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%BA%D0%B8%20%C2%AB%D0%9A%D0%B8%D0%BD%D0%BE%D1%82%D0%B5%D0%B0%D1%82%D1%80%20%D0%A6%D0%B5%D0%BD%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9%C2%BB.jpg'


Téléchargement et extraction:  31%|███       | 308/1000 [04:31<12:36,  1.09s/it]/Users/maxime/Documents/Git Projects/TraitementDonneeMassiveProjet/venv/lib/python3.13/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (99998272 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Téléchargement et extraction:  56%|█████▌    | 560/1000 [09:27<07:07,  1.03it/s]

Erreur lors du téléchargement : http://commons.wikimedia.org/wiki/Special:FilePath/%D9%BE%D8%A7%D9%86%D9%88%D8%B1%D8%A7%D9%85%D8%A7%DB%8C%20%D9%BE%D8%B1%D8%B3%D9%BE%DA%A9%D8%AA%DB%8C%D9%88%20%D8%A7%D8%B2%20%D9%86%D9%85%D8%A7%DB%8C%20%D8%AC%D9%86%D9%88%D8%A8%DB%8C%20%D8%AF%D8%B1%D9%88%D8%A7%D8%B2%D9%87%20%D8%A7%D8%B1%DA%AF%20%D8%B3%D9%85%D9%86%D8%A7%D9%86.jpg | [Errno 63] File name too long: '../images/%D9%BE%D8%A7%D9%86%D9%88%D8%B1%D8%A7%D9%85%D8%A7%DB%8C%20%D9%BE%D8%B1%D8%B3%D9%BE%DA%A9%D8%AA%DB%8C%D9%88%20%D8%A7%D8%B2%20%D9%86%D9%85%D8%A7%DB%8C%20%D8%AC%D9%86%D9%88%D8%A8%DB%8C%20%D8%AF%D8%B1%D9%88%D8%A7%D8%B2%D9%87%20%D8%A7%D8%B1%DA%AF%20%D8%B3%D9%85%D9%86%D8%A7%D9%86.jpg'


Téléchargement et extraction:  63%|██████▎   | 633/1000 [10:45<05:42,  1.07it/s]

Erreur lors du téléchargement : http://commons.wikimedia.org/wiki/Special:FilePath/Jabal%20Al%20Qara%20Cave%20-%20Al%20Hassa%2C%20Saudi%20Arabia%20%E0%B4%9C%E0%B4%AC%E0%B5%BD%20%E0%B4%85%E0%B5%BD%20%E0%B4%96%E0%B4%BE%E0%B4%B1%20%E0%B4%97%E0%B5%81%E0%B4%B9%2C%20%E0%B4%85%E0%B5%BD%20%E0%B4%B9%E0%B4%B8%2C%20%E0%B4%B8%E0%B5%97%E0%B4%A6%E0%B4%BF%20%E0%B4%85%E0%B4%B1%E0%B5%87%E0%B4%AC%E0%B5%8D%E0%B4%AF%2012.JPG | [Errno 63] File name too long: '../images/Jabal%20Al%20Qara%20Cave%20-%20Al%20Hassa%2C%20Saudi%20Arabia%20%E0%B4%9C%E0%B4%AC%E0%B5%BD%20%E0%B4%85%E0%B5%BD%20%E0%B4%96%E0%B4%BE%E0%B4%B1%20%E0%B4%97%E0%B5%81%E0%B4%B9%2C%20%E0%B4%85%E0%B5%BD%20%E0%B4%B9%E0%B4%B8%2C%20%E0%B4%B8%E0%B5%97%E0%B4%A6%E0%B4%BF%20%E0%B4%85%E0%B4%B1%E0%B5%87%E0%B4%AC%E0%B5%8D%E0%B4%AF%2012.JPG'


Téléchargement et extraction: 100%|██████████| 1000/1000 [17:22<00:00,  1.04s/it]
